In [ ]:
import time
import importlib
import cv2
import numpy as np
from ugot import ugot
import pose_yolo

# Reload the pose control module so any edits to pose_yolo.py take effect
# without restarting the kernel.
importlib.reload(pose_yolo)
from pose_yolo import run_pose_control_inline

# Utilities for showing live camera frames directly inside the notebook
from IPython.display import display, clear_output, Image
from PIL import Image as Image2

# Create the robot controller object
got = ugot.UGOT()

# Connect to the UGOT robot using its IP address.
# Change this string if your robot is on a different IP.
got.initialize("10.146.246.43")

# Load the built-in computer vision models that will be used later in the notebook.
# You can load additional models here if needed — just add the model name to the list.
got.load_models(
    [
        "color_recognition",  # detects dominant colors
        "word_recognition",  # OCR: reads printed text
        "line_recognition",  # for line-following tasks
        "face_recognition",  # identifies registered faces by name
    ]
)

# Select the default line-tracking mode.
# 0 = single-line mode (follows one line at a time)
got.set_track_recognition_line(0)

# Open the camera stream so later functions can read live frames
got.open_camera()

got.mechanical_joint_control(angle1=0, angle2=90, angle3=-120, duration=500)

try:
    run_pose_control_inline(
        robot_ip="10.146.246.43",
        forward_speed=20,
        backward_speed=20,
        turn_speed=40,
        camera_index=0,
        model_path="yolov8n-pose.pt",
        up_margin_factor=0.03,
        down_margin_factor=0.6,
        min_conf=0.3,
        enable_robot=True,  # <-- Robot is ENABLED: it will now move!
        debounce_frames=2,
        max_frames=None,
    )


except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done")

In [11]:
from ugot import ugot  # UGOT robot SDK
import time  # For sleep/timing between motor commands
import importlib
import cv2
import numpy as np
from IPython.display import display, clear_output, Image
from PIL import Image as Image2

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("10.146.246.43")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode", "face_recognition"]
)

# Open UGOT camera for later image recognition
got.open_camera()

Seen = False
Reached = False

def face_find_and_approach(gap, target_name, turn_spd, strafe_spd, fwd_spd, height):
    global Seen, Reached
    # Phase 1 - Find
    face_name = None


    while not Seen:
        got.mecanum_turn_speed(turn=3, speed=turn_spd)
        name = got.get_words_result()
        faces = got.get_face_recognition_total_info()

        if faces:
            for face in faces:
                if face[0] == target_name:
                    Seen = True
                    got.mecanum_stop()
                    print(f"Saw {target_name}!")
                    return

    # Phase 2 - Move
    while Seen and not Reached:
        name = got.get_words_result()
        faces = got.get_face_recognition_total_info()
        if not faces:
            # Lost the face; inch forward slowly to try to find it again
            got.mecanum_translate_speed(angle=0, speed=fwd_spd)
        else:
            c_x = faces[0][1]  # Horizontal center of the face in the frame (0–640 px)
            h = faces[0][3]  # Height of the face bounding box (proxy for distance)
            if h < height:
                if c_x < 320 - gap:
                    # Face is too far LEFT — strafe left while moving forward
                    got.mecanum_move_xyz(
                        x_speed=-strafe_spd, y_speed=fwd_spd, z_speed=0
                    )
                elif c_x > 320 + gap:
                    # Face is too far RIGHT — strafe right while moving forward
                    got.mecanum_move_xyz(x_speed=strafe_spd, y_speed=fwd_spd, z_speed=0)
                else:
                    # Face is centered but still small (far away) — move straight forward
                    got.mecanum_move_xyz(x_speed=0, y_speed=fwd_spd, z_speed=0)
            else:
                # Face is centered AND large enough — we've arrived!
                got.mecanum_stop()
                print(f"Reached {name}!")
                Reached = True
        # clear_output(wait=True)

    if Seen and Reached:
        got.mecanum_stop()
        return True

try:
    while True:
        got.mechanical_clamp_close()
        got.mechanical_joint_control(angle1=0, angle2=90, angle3=-90, duration=500)
        AtTarget = face_find_and_approach(gap=15, target_name="Arjun", turn_spd=15, strafe_spd=10, fwd_spd=10, height=120)
        if AtTarget:
            got.mecanum_translate_speed_times(angle=180, speed=10, times=10, unit=1)
            time.sleep(0.5)
            got.mechanical_joint_control(angle1=0, angle2=0, angle3=-60, duration=500)
            time.sleep(1)
            got.mechanical_clamp_release()
            got.mecanum_translate_speed_times(angle=180, speed=10, times=5, unit=1)
            break
except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done.")

10.146.246.43:50051
Saw Arjun!
Reached Arjun!
